# EgoLanes

## I. Getting Started Guide

### 1. EgoLanes Model Introduction

EgoLanes is a legacy semantic segmentation network originally developed for the VisionPilot suite to detect and segment ego-lane boundaries without relying on HD maps. It follows an Encoder-Context-Decoder pattern, optimized for spatial precision and global road awareness.

The architecture components include:

- **EfficientNet-B0 Backbone (Encoder)**: extracts hierarchical features at five different scales (`[l0, l2, l3, l4, l8]`), capturing everything from fine textures to broad semantic shapes.

- **Multi-scale Feature Fusion**: downsamples early high-resolution features to a uniform spatial size ($10 \times 20$ for the default $320 \times 640$ input) and concatenates them into a 1456-channel deep feature block.

- **AutoSteer Context Block**: compresses global spatial information through an MLP and projects it back into the feature maps using a residual attention mechanism ($Context \times Features + Features$).

- **EgoPath Neck (Decoder)**: upsamples the contextualized features back to 1/4 target resolution ($80 \times 160$) while using skip connections from the encoder to re-inject fine-grained spatial details.

- **EgoLanes Head**: 3 convolutional layers producing a 3-channel segmentation output (`ego-left lane`, `ego-right lane`, `other lanes`).

As a lane boundary detection model, EgoLanes processes raw image frames and performs real-time semantic segmentation of driving lanes in the image. It outputs lanes at 1/4 resolution of the input image size, allowing for quick inference on embedded hardware. EgoLanes was trained with data from real-world datasets including TuSimple, OpenLane, CurveLanes, Jiqing, and ONCE3D Lane.

![](Media/EgoLanes_GIF_2.gif)

### 2. Environment Setup

First, if you have not already cloned the repository locally, clone `ego_lanes` from the Autoware Foundation.

In [ ]:
import os

base_dir = os.getcwd()
print(base_dir)

# Clone ego_lanes repo if running outside the repository
if not os.path.exists("./Models"):
    if not os.path.exists("./ego_lanes"):
        !git clone https://github.com/autowarefoundation/ego_lanes.git
    %cd ego_lanes
    base_dir = os.getcwd()

The EgoLanes project relies on several key libraries, including **PyTorch** for model execution and **OpenCV** for image processing.

We install the required dependencies directly from `requirements.txt` at the repository root.

In [ ]:
# Install required dependencies
%pip install --upgrade pip
%pip install -r ./requirements.txt

# Verify the installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA availability: {torch.cuda.is_available()}")

### 3. Download Models

You can manually download the model weight files using the links below. Subsequent tutorial sections assume they are saved inside `./weights/`.

- [Link to Download PyTorch Model Weights (*.pth)](https://drive.google.com/file/d/1Njo9EEc2tdU1ffo8AUQ9mjwuQ9CzSRPX/view?usp=sharing)
- [Link to Download ONNX FP32 Model Weights (*.onnx)](https://drive.google.com/file/d/1b4jAoH6363ggTgVU0b6URbFfcOL3-r1Q/view?usp=sharing)

You can also run the cell below, which uses `gdown` to download the weights into `./weights/`.

In [ ]:
%pip install gdown

import os
import gdown

base_dir = os.getcwd()
model_dir = "./weights"
os.makedirs(model_dir, exist_ok=True)

metadata = {
    "pth": {
        "url": "https://drive.google.com/uc?id=1Njo9EEc2tdU1ffo8AUQ9mjwuQ9CzSRPX",
        "filepath": os.path.join(base_dir, model_dir, "egolanes.pth"),
    },
    "onnx": {
        "url": "https://drive.google.com/uc?id=1b4jAoH6363ggTgVU0b6URbFfcOL3-r1Q",
        "filepath": os.path.join(base_dir, model_dir, "egolanes.onnx"),
    },
}

for weight_type, info in metadata.items():
    url = info["url"]
    filepath = info["filepath"]
    if not os.path.exists(filepath):
        print(f"Downloading EgoLanes {weight_type} weights...")
        gdown.download(url, filepath, quiet=False)
    else:
        print(f"EgoLanes {weight_type} weights already exist at {filepath}. Skipping download.")

## II. Quick Test/Inference

Pre-requisites: please ensure that you have completed the **Environment Setup** and **Download Models** steps above.

### 1. Image Inference

You can run inference directly via `EgoLanesNetworkInfer` (PyTorch) or `EgoLanesONNXInfer` (ONNX Runtime), or batch-process a directory of images using `Models/visualizations/image_visualization.py`.

#### a. Run Batch Image Inference

Key script parameters for `Models/visualizations/image_visualization.py`:
- `-p / --model_checkpoint_path`: path to your `.pth` or `.onnx` weights file.
- `-i / --input_image_dirpath`: image file or directory containing `.png`, `.jpg`, or `.jpeg` files.
- `-o / --output_image_dirpath`: destination directory where visualization files (`[original_id]_data.png`) will be saved.

In [ ]:
CHECKPOINT = metadata["pth"]["filepath"]
INPUT_DIR = os.path.join(base_dir, "Media")
OUTPUT_DIR = os.path.join(base_dir, "results", "images")

!python3 Models/visualizations/image_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_DIR)} \
    -o {os.path.abspath(OUTPUT_DIR)}

#### b. Display Results in Notebook

After running inference, we can use matplotlib and PIL to visualize the saved results directly within this notebook.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

result_files = sorted([
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith("_data.png")
])

if result_files:
    fig, axes = plt.subplots(1, 3, figsize=(20, 10))
    for i, ax in enumerate(axes):
        if i < len(result_files):
            img_path = os.path.join(OUTPUT_DIR, result_files[i])
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"Result: {result_files[i]}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No result images found. Check your output directory.")

### 2. Video Inference

In this subsection, we apply the EgoLanes model to video streams using `Models/visualizations/video_visualization.py`, which processes video frames with OpenCV, performs lane segmentation (supporting both `.pth` and `.onnx` models), and writes an annotated `.avi` video.

Key script parameters:

- `-p / --model_checkpoint_path`: path to the trained EgoLanes `.pth` or `.onnx` weights.
- `-i / --input_video_filepath`: path to the source video file.
- `-o / --output_video_path`: destination path for the visualized video (`.avi`).
- `-v / --vis`: optional flag to enable a pop-up window showing frames as they are processed.

Technical details:

- Input frames are converted from BGR to RGB and resized to $(640, 320)$ for inference, then upscaled to $(720, 360)$ for the final MJPG `.avi` output.
- Video capture and writer resources are automatically released on completion.

In [ ]:
INPUT_PATH = os.path.join(base_dir, "Media", "highway_normal.mp4")
OUTPUT_PATH = os.path.join(base_dir, "results", "videos", "highway_normal_output.avi")

!python3 Models/visualizations/video_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_PATH)} \
    -o {os.path.abspath(OUTPUT_PATH)}

## III. Model Training

### 1. Model Training on New Data

#### a. Load pre-trained model or train from scratch

Training is launched from `Models/training/train_ego_lanes.py`, which initializes `EgoLanesTrainer`:

- Without `--checkpoint_path`, `EgoLanesTrainer()` initializes `EgoLanesNetwork` with an ImageNet-pretrained EfficientNet-B0 backbone and trains from scratch.
- With `--checkpoint_path /path/to/checkpoint.pth`, `EgoLanesTrainer(checkpoint_path=...)` loads existing weights to resume or fine-tune.

Practical recommendation:
- Start from scratch when your datasets or lane conventions differ significantly.
- Start from a checkpoint when you want faster convergence and stable early metrics.

#### b. Prepare the training datasets

EgoLanes training data is generated with the scripts under `Models/data_parsing/`.

Each parser converts raw annotations into a unified format:

```
--output_dir
    ├── image
    ├── mask
    ├── visualization
    └── drivable_path.json
```

Most scripts **purge `--output_dir` if it already exists**. Use a fresh output path.
You can pass `--early_stopping` for quick smoke tests before running a full dataset conversion.

##### i) TuSimple

```bash
python3 Models/data_parsing/TuSimple/process_tusimple.py \
  --dataset_dir /path/to/TuSimple \
  --output_dir /path/to/processed/TUSIMPLE
```

##### ii) CurveLanes

```bash
python3 Models/data_parsing/CurveLanes/process_curvelanes.py \
  --dataset_dir /path/to/CurveLanes \
  --output_dir /path/to/processed/CURVELANES \
  --sampling_step 5
```

##### iii) OpenLane

```bash
python3 Models/data_parsing/OpenLane/process_openlane.py \
  --dataset_dir /path/to/OpenLane \
  --output_dir /path/to/processed/OPENLANE
```

##### iv) Jiqing (Jinan-Qingdao Expressway)

```bash
python3 Models/data_parsing/Jiqing/process_jiqing.py \
  --video_dir /path/to/Jiqing_Videos \
  --gt_dir /path/to/Lane_Parameters \
  --output_dir /path/to/processed/JIQING
```

##### v) Once3DLane

```bash
python3 Models/data_parsing/Once3DLane/process_once3d.py \
  --dataset_dir /path/to/Once3DLane \
  --output_dir /path/to/processed/ONCE3DLANE
```

#### c. Expected processed dataset layout

`train_ego_lanes.py` loads processed datasets using `LoadDataEgoLanes` from `--dataset_root`:

```
<DATASET_ROOT>/
├── TUSIMPLE/
│   ├── image/
│   ├── mask/
│   ├── visualization/
│   └── drivable_path.json
├── CURVELANES/
├── OPENLANE/
├── JIQING/
└── ONCE3DLANE/
```

Before full training, ensure frame IDs in `drivable_path.json` match image filenames in `image/` and mask counts equal image counts.

#### d. How to run training

From the repository root:

```bash
python3 Models/training/train_ego_lanes.py \
  --dataset_root /path/to/processed_datasets_root \
  --datasets CURVELANES TUSIMPLE \
  --save_dir runs/ego_lanes \
  --epochs 10 \
  --batch_size 8 \
  --lr 1e-4
```

During training, the script:
- Loads each selected dataset via `LoadDataEgoLanes`.
- Applies `Augmentations(data_type="SEGMENTATION")`.
- Computes weighted BCE + multi-scale edge preservation loss in `EgoLanesTrainer`.
- Uses gradient accumulation every `--batch_size` samples.
- Saves checkpoints to `<save_dir>/models/` and validation figures to `<save_dir>/figures/`.

#### e. How to visualize training results

This training pipeline provides two visualization channels:

1. **TensorBoard logs** (loss curves + train visualization panels)
2. **Saved validation figures** (`*_seg.png`, `*_seg_raw.png`) under `<save_dir>/figures/`

![Example of Overall Validation (TensorBoard)](Media/overall_val_tensorboard_example.png)

Run TensorBoard from the repository root:

```bash
tensorboard --logdir runs --port 6006
```

Then open `http://localhost:6006` in your browser.

### 2. Exporting to ONNX

To export a trained `.pth` checkpoint to FP32 `.onnx`:

```bash
python3 Models/exports/convert_pytorch_to_onnx.py \
  -p weights/egolanes.pth \
  -o weights/egolanes.onnx
```